In [ ]:
# =============================================================
# CELL 1 — Config
# =============================================================

# DBFS Folder
pdf_folder = "your_folder"

# N8N Webhook URL
n8n_webhook_url = "your_n8n_webhook_url"

# Gemini API Key
gemini_api_key = "your_gemini_API_key"

# Embedding model Gemini
GEMINI_EMBED_URL = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    "gemini-embedding-2:embedContent"
)

chunk_size = 1200
chunk_overlap = 200

print("Config loaded.")
print(f"PDF folder : {pdf_folder}")
print(f"Webhook    : {n8n_webhook_url}")
print(f"Embed model: gemini-embedding-2 (768 dim)")

Config loaded.
PDF folder : /Volumes/dbacademy/default/raw_data/policies/
Webhook    : https://niribal.my.id/webhook/policy-ingest
Embed model: gemini-embedding-2 (768 dim)


In [ ]:
# =============================================================
# CELL 2 — Helper: chunk text dengan overlap
# =============================================================

def chunk_text(text: str, chunk_size: int = chunk_size, overlap: int = chunk_overlap) -> list:
    """"
    Chunk text into smaller chunks with overlap.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) >= 50]

print(f"Chunk size: {chunk_size} chars, overlap: {chunk_overlap} chars")
    


Chunk size: 1200 chars, overlap: 200 chars


In [ ]:
# =============================================================
# CELL 3 — Helper: generate embedding via Gemini API
# =============================================================
import requests
import time

def get_embedding (text: str, retries: int = 3) -> list:
    """
    Send a text to Gemini gemini-embedding-2, return vector 768 dimension.
    Retry Automatically if rate limit (15 req/menit).
    """
    payload = {
        "model": "gemini-embedding-2",
        "content": {"parts": [{"text": text}]},
        "outputDimensionality": 768
    }

    for attempt in range(retries):
        resp = requests.post(
            f"{GEMINI_EMBED_URL}?key={gemini_api_key}",
            json=payload,
            timeout=30,
        )

        if resp.status_code == 200:
            return resp.json()["embedding"]["values"]
        elif resp.status_code == 429:
            wait = 5 * (attempt + 1)
            print(f"Rate limit detected, {wait}s...")
            time.sleep(wait)
        else:
            raise Exception(f"Gemini API error {resp.status_code}: {resp.text}")
    raise Exception("Failed to generate embedding after a few trials.")

def get_embeddings_batch(texts: list, delay: float = 0.1) -> list:
    """
    Generate embedding for each text in the list.
    Delay between requests to prevent rate limit.
    """
    embeddings = []
    total = len(texts)

    for i, text in enumerate(texts):
        if i % 10 == 0:
            print (f"   Embedding : {i+1}/{total}.....")
        embeddings.append(get_embedding(text))
        time.sleep(delay)
    return embeddings

print("Embedding helper loaded.")


Embedding helper loaded.


In [ ]:
# =============================================================
# CELL 4 — Parse all PDF from folder
# =============================================================
import pdfplumber
import os

def parse_pdf(filepath: str) -> list:
    """
    Parse PDF file and return list of chunks.
    """
    filename = os.path.basename(filepath)
    topic = os.path.splitext(filename)[0].replace("_", " ").title()
    results = []

    with pdfplumber.open(filepath) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text()
            if not page_text:
                continue

            page_text = " ".join(page_text.split())
            chunks = chunk_text(page_text)

            for chunk_idx, chunk in enumerate(chunks):
                results.append({
                    "chunk_id"   : f"{filename}_p{page_num}_c{chunk_idx}",
                    "source_file": filename,
                    "topic"      : topic,
                    "page"       : page_num,
                    "chunk_index": chunk_idx,
                    "text"       : chunk,
                })

    return results

#Parse all PDF
all_chunks = []
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF files: {pdf_files}\n")

for pdf_file in pdf_files:
    filepath = os.path.join(pdf_folder, pdf_file)
    chunks   = parse_pdf(filepath)
    all_chunks.extend(chunks)
    print(f"  {pdf_file} → {len(chunks)} chunks")
 
print(f"\nTotal chunks: {len(all_chunks)}")

Found 4 PDF files: ['company_policy_manual_sample.pdf', 'health_and_safety_policy_sample.pdf', 'it_and_security_policy_sample.pdf', 'remote_work_policy_sample.pdf']

  company_policy_manual_sample.pdf → 16 chunks
  health_and_safety_policy_sample.pdf → 22 chunks
  it_and_security_policy_sample.pdf → 23 chunks
  remote_work_policy_sample.pdf → 20 chunks

Total chunks: 81


In [ ]:
# =============================================================
# CELL 6 — Generate embeddings via Gemini
# =============================================================
# Estimated Time: ~100 chunks = ~1 minute (delay 0.1s/request)
# Free tier limit: 1.500 req/day

print("Generating embeddings via Gemini gemini-embedding-2 ...")
print(f"Estimated time: ~{len(all_chunks) * 0.1 / 60:.1f} minute\n")

texts = [c["text"] for c in all_chunks]
embeddings = get_embeddings_batch(texts, delay=0.1)

for i, chunk in enumerate(all_chunks):
    chunk["embedding"] = embeddings[i]

print(f"\nDone. Embedding dimension: {len(all_chunks[0]['embedding'])}")


Generating embeddings via Gemini gemini-embedding-2 ...
Estimated time: ~0.1 minute

   Embedding : 1/81.....
   Embedding : 11/81.....
   Embedding : 21/81.....
Rate limit detected, 5s...
   Embedding : 31/81.....
   Embedding : 41/81.....
   Embedding : 51/81.....
   Embedding : 61/81.....
   Embedding : 71/81.....
   Embedding : 81/81.....

Done. Embedding dimension: 768


In [ ]:
# =============================================================
# CELL 7 — Simpan ke Delta table
# =============================================================
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, ArrayType
)

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("chunk_id", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("topic", StringType(), True),
    StructField("page", IntegerType(), True),
    StructField("chunk_index", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("embedding", ArrayType(FloatType()), True),
])

df = spark.createDataFrame(all_chunks, schema=schema)

TABLE_NAME = "policy_chunks"
df.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

print(f"Saved {df.count()} rows to Delta Table: {TABLE_NAME}")
df.select("chunk_id", "topic", "page", "chunk_index").show(10, truncate=False)

Saved 81 rows to Delta Table: policy_chunks
+--------------------------------------+----------------------------+----+-----------+
|chunk_id                              |topic                       |page|chunk_index|
+--------------------------------------+----------------------------+----+-----------+
|company_policy_manual_sample.pdf_p1_c0|Company Policy Manual Sample|1   |0          |
|company_policy_manual_sample.pdf_p1_c1|Company Policy Manual Sample|1   |1          |
|company_policy_manual_sample.pdf_p2_c0|Company Policy Manual Sample|2   |0          |
|company_policy_manual_sample.pdf_p2_c1|Company Policy Manual Sample|2   |1          |
|company_policy_manual_sample.pdf_p2_c2|Company Policy Manual Sample|2   |2          |
|company_policy_manual_sample.pdf_p3_c0|Company Policy Manual Sample|3   |0          |
|company_policy_manual_sample.pdf_p3_c1|Company Policy Manual Sample|3   |1          |
|company_policy_manual_sample.pdf_p4_c0|Company Policy Manual Sample|4   |0          |

In [ ]:
# =============================================================
# CELL 8 — (Optional) Result Preview
# =============================================================
spark.sql(f"""
    SELECT chunk_id, topic, page, LEFT(text, 120) AS preview
    FROM {TABLE_NAME}
    ORDER BY topic, page, chunk_index
    LIMIT 20
""").show(truncate=False)

+-----------------------------------------+-------------------------------+----+------------------------------------------------------------------------------------------------------------------------+
|chunk_id                                 |topic                          |page|preview                                                                                                                 |
+-----------------------------------------+-------------------------------+----+------------------------------------------------------------------------------------------------------------------------+
|company_policy_manual_sample.pdf_p1_c0   |Company Policy Manual Sample   |1   |COMPANY POLICY MANUAL SAMPLE LTD Company Name: Sample Ltd Effective Date: June 3, 2026 Version: 1.0 Applicability: All E|
|company_policy_manual_sample.pdf_p1_c1   |Company Policy Manual Sample   |1   |with or without prior notice, subject to applicable law. Employees are responsible for reviewing updates and see

In [ ]:
# =============================================================
# CELL 9 — Send to n8n via webhook
# =============================================================
import json
 
def send_to_n8n(chunks: list, webhook_url: str, batch_size: int = 50):
    """
    Send chunks to n8n in batch to prevent timeout.
    """
    total   = len(chunks)
    batches = [chunks[i:i+batch_size] for i in range(0, total, batch_size)]
    print(f"Sending {total} chunks in {len(batches)} batches to n8n ...")
 
    for idx, batch in enumerate(batches):
        payload = {
            "batch_number" : idx + 1,
            "total_batches": len(batches),
            "total_chunks" : total,
            "chunks"       : batch,
        }
        response = requests.post(
            webhook_url,
            headers={"Content-Type": "application/json"},
            data=json.dumps(payload),
            timeout=30,
        )
        if response.status_code == 200:
            print(f"  Batch {idx+1}/{len(batches)} OK")
        else:
            print(f"  Batch {idx+1} FAILED: {response.status_code} — {response.text}")
 
send_to_n8n(all_chunks, n8n_webhook_url)
print("\nPipeline complete!")


Sending 81 chunks in 2 batches to n8n ...
  Batch 1/2 OK
  Batch 2/2 OK

Pipeline complete!
